# Inferência de crédito com CKKS-ML

**Status científico:** EXPLORATÓRIO. Este notebook compara uma árvore de decisão treinada em claro com a mesma árvore adaptada ao CKKS-ML, sobre as mesmas observações da Freddie Mac. Ele aceita um run N01 privado e verificado ou uma **reprodução local provisória** do protocolo N01. A reprodução local não é automaticamente o mesmo modelo ou conjunto de dados usado pela Kethlen.

O objetivo é avaliar fidelidade numérica, decisões simuladas e custo da inferência. A execução real cifra as entradas antes do cálculo; a geração de chaves, a cifragem e a decifragem ocorrem no ambiente local. A simulação não executa criptografia real. Cinco observações cifradas são uma verificação inicial, não evidência de desempenho populacional.


## 1. Referência e privacidade

Escolha explicitamente uma origem:

- **`run`**: usa um Run ID N01 concluído e de origem confiável, com `manifest.json`, `effective_config.json`, `environment.json`, `prepared_dataset.npz` e `sklearn_model.joblib`. O notebook verifica o manifesto e os hashes; isso detecta alterações em relação ao manifesto, mas não autentica sua origem. Um arquivo `joblib` pode executar código: carregue apenas modelos confiáveis.
- **`reproducao_local`**: lê os ZIPs oficiais locais indicados no `config.yaml` N01 e reconstrói em memória o alvo, as partições, o pré-processamento e a árvore com o código da Kethlen. Registra hashes e versões, sem criar nem simular um manifesto de run concluído. A Freddie Mac é uma base que pode ser atualizada: sem comparar os hashes de entrada e o ambiente da execução original, esta opção não comprova que os dados ou o modelo sejam idênticos aos dela.

O N01 explora sete variáveis de originação, alvo de atraso observado e safras 2015–2017 para treino, 2019 para validação e 2021 para teste. A calibração usa até 1.000 linhas de treino selecionadas com semente 42. Por padrão a avaliação ampliada usa até as primeiras 10.000 linhas de teste, como o N01 inicial; configure `MAX_ANALYSIS_SAMPLES=None` para avaliar todo o teste, como algumas tabelas do artigo. A FHE usa as primeiras cinco. Isso pareia observações, mas as primeiras linhas não são necessariamente representativas.

Mantenha dados, modelos, matrizes, chaves e resultados individuais privados. Não salve saídas do notebook no Git; métricas agregadas também exigem revisão antes de qualquer publicação.


In [ ]:
from pathlib import Path
import json
import re
import sys

# Execute na raiz do projeto ou em uma pasta dentro dele.
working_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'experiments').is_dir() and (path / 'src' / 'fhe_credit').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Abra o notebook a partir da raiz do repositório ou de experiments/.')

REFERENCE_SOURCE = 'reproducao_local'  # 'run' ou 'reproducao_local'; sem troca automática.
N01_RUN_ID = None  # obrigatório na opção 'run'; use somente um run confiável.
DATA_ROOT = project_root / 'datasets'  # ZIPs oficiais locais e privados, para 'reproducao_local'.
REBUILD_SKLEARN_VERSION = '1.5.0'  # versão declarada no lock FHE do repositório; confira o ambiente original.
CKKS_N_BITS = 6  # precisão das entradas da árvore CKKS; não equivale ao n_bits do Concrete ML.
MAX_ANALYSIS_SAMPLES = 10000  # mesmas primeiras 10 mil do N01; None usa todo o teste.
POLICY_CUTOFF = None  # opcional: limiar de aprovação definido previamente em validação.
RUN_FHE_EXECUTE = True  # cinco observações; a primeira chamada pode ser lenta.

if REFERENCE_SOURCE not in {'run', 'reproducao_local'}:
    raise ValueError("REFERENCE_SOURCE deve ser 'run' ou 'reproducao_local'.")
if REFERENCE_SOURCE == 'run' and (not isinstance(N01_RUN_ID, str) or not re.fullmatch(r'run-[A-Za-z0-9_-]{3,100}', N01_RUN_ID)):
    raise ValueError('Defina N01_RUN_ID com o identificador de um run N01 privado e confiável.')
if type(CKKS_N_BITS) is not int or CKKS_N_BITS < 2:
    raise ValueError('CKKS_N_BITS deve ser um inteiro de pelo menos 2 bits.')
if MAX_ANALYSIS_SAMPLES is not None and (type(MAX_ANALYSIS_SAMPLES) is not int or MAX_ANALYSIS_SAMPLES < 1):
    raise ValueError('MAX_ANALYSIS_SAMPLES deve ser None ou um inteiro positivo.')
if POLICY_CUTOFF is not None and (type(POLICY_CUTOFF) not in (int, float) or not 0 <= POLICY_CUTOFF <= 1):
    raise ValueError('POLICY_CUTOFF deve ser None ou um valor numérico entre 0 e 1.')
if type(RUN_FHE_EXECUTE) is not bool:
    raise ValueError('RUN_FHE_EXECUTE deve ser True ou False.')

unit = project_root / 'experiments' / '04_numerical_fidelity' / 'N01_freddie_mac_decision_tree'
run_dir = unit / 'results' / 'runs' / N01_RUN_ID if REFERENCE_SOURCE == 'run' else None
if run_dir is not None and not run_dir.is_dir():
    raise FileNotFoundError('Run N01 privado não encontrado. Selecione outro run ou a reprodução local explícita.')


## 2. Obter e verificar a referência

Para um run, verificamos `COMPLETED`, a identidade do experimento, o hash da configuração, todos os artefatos registrados e a versão scikit-learn do ambiente que treinou a árvore. Para a reprodução local, usamos os ZIPs oficiais exigidos pelo N01, a função de leitura e a sequência de pré-processamento do repositório; dados e derivados permanecem em memória. Nessa opção, a versão scikit-learn é fixada por padrão à declarada no lock FHE do projeto, o que não comprova, sozinho, o ambiente efetivo da execução original.

CKKS-ML 0.1.0 requer Linux x86-64 e Python 3.12. Em Apple Silicon, use um ambiente compatível ou emulado e registre essa condição na interpretação dos tempos.


In [ ]:
import importlib
import platform
import subprocess
import time
from importlib import metadata

import joblib
import numpy as np
import sklearn
from sklearn.tree import DecisionTreeClassifier as SklearnTree
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from ckksml.sklearn import DecisionTreeClassifier as CKKSTree

sys.path.insert(0, str(project_root / 'src'))
import fhe_credit
if Path(fhe_credit.__file__).resolve().parent != (project_root / 'src' / 'fhe_credit').resolve():
    raise RuntimeError('O kernel carregou outro repositório; reinicie-o a partir deste projeto.')
from fhe_credit.configuration import config_hash, load_config
from fhe_credit.run_manifest import load_manifest, sha256_file, verify_artifact_hashes

expected_features = ['credit_score', 'original_ltv', 'original_cltv', 'original_dti',
                     'original_upb', 'original_loan_term', 'number_of_borrowers']
expected_splits = {'training_vintages': [2015, 2016, 2017], 'validation_vintages': [2019], 'test_vintages': [2021]}

def check_protocol(config):
    if config['experiment_id'] != 'n01-freddie-mac-decision-tree':
        raise ValueError('A configuração não pertence ao experimento N01.')
    parameters = config['parameters']
    if parameters.get('features') != expected_features or any(parameters.get(k) != v for k, v in expected_splits.items()):
        raise ValueError('Ordem de variáveis ou partições diferem do protocolo N01 esperado.')
    if parameters.get('seed') != 42 or parameters.get('horizon_months') != 12 or parameters.get('target_scenario') != 'S01':
        raise ValueError('Semente, horizonte ou alvo diferem do protocolo N01 esperado.')
    return parameters

if REFERENCE_SOURCE == 'run':
    manifest = load_manifest(run_dir)
    required = {'effective_config.json', 'environment.json', 'prepared_dataset.npz', 'sklearn_model.joblib'}
    if manifest['run_id'] != N01_RUN_ID or manifest['experiment_id'] != 'n01-freddie-mac-decision-tree':
        raise ValueError('O manifesto não pertence ao Run ID e experimento N01 informados.')
    if manifest['status'] != 'COMPLETED':
        raise ValueError('A referência N01 precisa estar concluída antes da comparação.')
    if not required.issubset(set(manifest['required_artifacts'])) or not required.issubset(manifest['artifact_hashes']):
        raise ValueError('O manifesto não registra todos os artefatos necessários.')
    if not verify_artifact_hashes(run_dir, manifest):
        raise ValueError('Um ou mais artefatos não correspondem aos hashes do manifesto.')
    config = json.loads((run_dir / 'effective_config.json').read_text(encoding='utf-8'))
    if config_hash(config) != manifest['config_hash']:
        raise ValueError('A configuração efetiva não corresponde ao manifesto N01.')
    params = check_protocol(config)
    source_environment = json.loads((run_dir / 'environment.json').read_text(encoding='utf-8'))
    source_sklearn = sorted({p['version'] for p in source_environment.get('packages', [])
                             if p.get('name', '').lower().replace('_', '-') == 'scikit-learn'})
    if len(source_sklearn) != 1 or source_sklearn[0] != sklearn.__version__:
        raise RuntimeError('A versão scikit-learn desta execução não corresponde de forma inequívoca à versão do run N01.')
    # Arquivos verificados e confiáveis; as matrizes já foram pré-processadas no N01.
    with np.load(run_dir / 'prepared_dataset.npz', allow_pickle=False) as data:
        X_train, X_test, y_test = data['X_train'], data['X_test'], data['y_test']
    baseline_model = joblib.load(run_dir / 'sklearn_model.joblib')
    provenance = {'origem': 'run N01 verificado', 'run_id': N01_RUN_ID,
                  'config_hash': manifest['config_hash'], 'dataset_hash': manifest.get('dataset_hash'),
                  'modelo_sha256': manifest['artifact_hashes']['sklearn_model.joblib'],
                  'matrizes_sha256': manifest['artifact_hashes']['prepared_dataset.npz']}
else:
    import pandas as pd
    from fhe_credit.experiments import freddie_tree as n01_module

    n01_module = importlib.reload(n01_module)
    source_code = project_root / 'src' / 'fhe_credit' / 'experiments' / 'freddie_tree.py'
    if Path(n01_module.__file__).resolve() != source_code.resolve():
        raise RuntimeError('O código N01 carregado não corresponde a este repositório.')

    config_path = unit / 'config.yaml'
    config = load_config(config_path)
    params = check_protocol(config)
    if sklearn.__version__ != REBUILD_SKLEARN_VERSION:
        raise RuntimeError('Alinhe scikit-learn à versão escolhida para reproduzir N01 antes de continuar.')
    required_inputs = {item.get('role', item['path']): Path(DATA_ROOT) / Path(item['path']).name
                       for item in config['input_artifacts'] if item.get('required', True)}
    missing = sorted(role for role, path in required_inputs.items() if not path.is_file())
    if missing:
        raise FileNotFoundError('Arquivos oficiais locais ausentes para N01: ' + ', '.join(missing))
    hashes = {role: sha256_file(path) for role, path in sorted(required_inputs.items())}
    try:
        git_commit = subprocess.run(['git', '-C', str(project_root), 'rev-parse', 'HEAD'],
                                    capture_output=True, text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        # Em contêineres sem git, leia somente HEAD e a referência local correspondente.
        git_commit = 'indisponível'
        git_dir = project_root / '.git'
        try:
            head = (git_dir / 'HEAD').read_text(encoding='utf-8').strip()
            if head.startswith('ref: refs/') and '..' not in Path(head[5:]).parts:
                ref = head[5:]
                ref_file = git_dir / ref
                if ref_file.is_file():
                    head = ref_file.read_text(encoding='utf-8').strip()
                elif (git_dir / 'packed-refs').is_file():
                    head = next((line.split(' ', 1)[0] for line in (git_dir / 'packed-refs').read_text(encoding='utf-8').splitlines()
                                 if line.endswith(' ' + ref)), '')
            if re.fullmatch(r'[0-9a-f]{40,64}', head):
                git_commit = head
        except OSError:
            pass
    provenance = {'origem': 'reprodução local provisória; não é o run original',
                  'git_commit': git_commit, 'codigo_n01_sha256': sha256_file(source_code),
                  'config_hash': config_hash(config), 'entradas_sha256': hashes}
    frames = {}
    for year in [2015, 2016, 2017, 2019, 2021]:
        print(f'Preparando safra {year} em memória...')
        frame, _ = n01_module._load_vintage(required_inputs[f'sample_{year}'], year, params['horizon_months'],
                                 expected_features, set(params['missing_codes']), lambda _: None)
        frames[year] = frame
    train = pd.concat([frames[y] for y in params['training_vintages']], ignore_index=True)
    validation, test = frames[2019], frames[2021]
    # Mesma sequência N01, inclusive o recálculo dos quantis após a alteração da matriz de treino.
    medians = train[expected_features].median().to_dict()
    for frame in (train, validation, test):
        frame.loc[:, expected_features] = frame[expected_features].fillna(medians).clip(
            lower=train[expected_features].quantile(.001), upper=train[expected_features].quantile(.999), axis=1)
    X_train, y_train = train[expected_features].to_numpy(np.float32), train.target.to_numpy(int)
    X_test, y_test = test[expected_features].to_numpy(np.float32), test.target.to_numpy(int)
    baseline_model = SklearnTree(**params['model']).fit(X_train, y_train)
    del frames, train, validation, test, y_train

if not isinstance(baseline_model, SklearnTree) or not np.array_equal(baseline_model.classes_, [0, 1]):
    raise ValueError('Esperava-se uma árvore scikit-learn binária com classes [0, 1].')
if any(baseline_model.get_params().get(k) != v for k, v in params['model'].items()):
    raise ValueError('Os parâmetros da árvore não correspondem à configuração efetiva.')
if X_train.ndim != 2 or X_test.ndim != 2 or X_train.shape[1] != len(expected_features) or X_test.shape[1] != len(expected_features):
    raise ValueError('As matrizes devem ser bidimensionais e ter as sete colunas do N01.')
if len(X_train) == 0 or len(X_test) == 0 or y_test.ndim != 1 or len(y_test) != len(X_test):
    raise ValueError('As matrizes de treino, teste e rótulos devem estar completas e alinhadas.')
if not np.isfinite(X_train).all() or not np.isfinite(X_test).all() or not np.isin(y_test, [0, 1]).all():
    raise ValueError('As entradas devem ser finitas e os rótulos devem pertencer a [0, 1].')
if getattr(baseline_model, 'n_features_in_', None) != len(expected_features):
    raise ValueError('O número de variáveis da árvore não corresponde ao N01.')

def version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return 'indisponível'

calibration_size = min(params['calibration_sample_size'], len(X_train))
calibration_indices = np.random.default_rng(params['seed']).choice(len(X_train), calibration_size, replace=False)
X_calibration = X_train[calibration_indices]
wide_size = len(X_test) if MAX_ANALYSIS_SAMPLES is None else min(MAX_ANALYSIS_SAMPLES, len(X_test))
fhe_size = min(params['fhe_sample_size'], len(X_test))
if calibration_size < 1 or wide_size < 1 or fhe_size < 1 or fhe_size > wide_size:
    raise ValueError('Os tamanhos das amostras não são válidos para o pareamento.')
print({'proveniencia': provenance, 'python': platform.python_version(),
       'sistema': platform.system(), 'arquitetura': platform.machine(),
       'numpy': np.__version__, 'pandas': version('pandas'), 'scikit_learn': sklearn.__version__,
       'ckksml': version('ckksml'), 'openfhe': version('openfhe'),
       'calibracao_treino': calibration_size, 'teste_ampliado': wide_size,
       'teste_fhe': fhe_size, 'bits_entrada_ckks': CKKS_N_BITS})


## 3. Adaptar a mesma árvore ao CKKS-ML

A árvore é treinada em claro na reprodução local ou carregada de um run existente; depois, ela é adaptada ao CKKS-ML. Não há treinamento cifrado. Para árvores na versão testada do CKKS-ML, `n_bits` controla a quantização das **entradas** do programa CKKS. Esse parâmetro não deve ser considerado equivalente ao `n_bits` do Concrete ML, que tem outra implementação. A calibração pode limitar valores de teste ao intervalo observado no treino.

`disable` chama a árvore scikit-learn em claro; não representa uma árvore CKKS quantizada. `simulate` avalia o programa adaptado e quantizado sem criptografia; `execute` avalia esse programa com criptografia real. Na configuração padrão da árvore, o caminho e a saída da folha são avaliados de forma cifrada; após a decifragem, a biblioteca normaliza as probabilidades em claro.


In [ ]:
started = time.perf_counter()
model = CKKSTree.from_sklearn_model(baseline_model, X=X_calibration, n_bits=CKKS_N_BITS)
preparation_seconds = time.perf_counter() - started
if not np.array_equal(model.classes_, baseline_model.classes_):
    raise ValueError('A ordem das classes no CKKS-ML não corresponde à referência.')
print({'preparacao_total_s': preparation_seconds, 'compilacao_s': getattr(model, 'compile_time_', None),
       'bits_entrada_ckks': model.n_bits, 'saida_fhe': model.fhe_output,
       'limita_entradas': model.clip_inputs, 'margem_entrada': model.input_margin,
       'packing': model.packing})


## 4. Comparar probabilidades e decisões

As probabilidades da classe 1 são comparadas nas mesmas linhas e na mesma ordem. Para o conjunto ampliado, calculamos diferenças e métricas preditivas agregadas quando houver as duas classes. A regra `p >= 0,5` reproduz somente a verificação de concordância usada no N01; não é uma política de aprovação validada. Quando `p` é exatamente `0,5`, essa regra pode divergir da classe retornada por `predict()` devido ao desempate.

Também medimos a maior taxa de discordância decisória ao percorrer os possíveis limiares entre 0 e 1, seguindo a ideia do artigo. **Esse máximo não é a taxa observada em um limiar operacional**: mesmo diferenças pequenas podem afetar muitos empréstimos em uma faixa muito estreita. A versão ponderada usa o saldo original tratado como peso, não como perda financeira. Opcionalmente, informe `POLICY_CUTOFF` escolhido previamente em validação para medir uma regra específica.

Para as cinco observações FHE, executamos novamente referência, `disable` e simulação no mesmo lote usado em `execute`, evitando interpretar tempos de tamanhos de lote diferentes como comparáveis. As tabelas mostram apenas agregados, sem linhas individuais; os dicionários `wide_summary` e `small_summary` mantêm os valores calculados sem arredondamento de apresentação. Não compare diretamente métricas calculadas em 10.000 observações com as calculadas sobre todo o teste.


In [ ]:
def timed_probability(predict_fn, data):
    started = time.perf_counter()
    probabilities = np.asarray(predict_fn(data), dtype=float)
    elapsed = time.perf_counter() - started
    if probabilities.shape != (len(data), 2) or not np.isfinite(probabilities).all():
        raise ValueError('A saída deve ter duas probabilidades finitas por observação.')
    if (probabilities < 0).any() or (probabilities > 1).any() or not np.allclose(probabilities.sum(axis=1), 1, rtol=0, atol=1e-5):
        raise ValueError('Saída inválida: probabilidades devem estar em [0, 1] e somar 1 por observação.')
    return probabilities[:, 1], elapsed


def decision_disagreement(reference, values, weights=None):
    # A decisão p <= limiar difere apenas entre as duas probabilidades de cada linha.
    weights = np.ones(len(reference), dtype=float) if weights is None else np.asarray(weights, dtype=float)
    if weights.shape != reference.shape or not np.isfinite(weights).all() or (weights < 0).any() or weights.sum() <= 0:
        raise ValueError('Os pesos devem ser finitos, não negativos e ter soma positiva.')
    boundaries = np.unique(np.r_[0.0, 1.0, reference, values])
    lower, upper = np.minimum(reference, values), np.maximum(reference, values)
    events = np.zeros(len(boundaries), dtype=float)
    np.add.at(events, np.searchsorted(boundaries, lower), weights)
    np.add.at(events, np.searchsorted(boundaries, upper), -weights)
    rates = np.cumsum(events[:-1]) / weights.sum()
    area = float(np.dot(rates, np.diff(boundaries)))
    weighted_mae = float(np.average(np.abs(values - reference), weights=weights))
    return {'discordancia_maxima': float(np.max(rates)), 'area_discordancia': area,
            'diferenca_area_vs_mae': float(area - weighted_mae)}


def fidelity(reference, values, weights):
    delta = values - reference
    result = {'mae_vs_referencia': float(np.mean(np.abs(delta))),
              'rmse_vs_referencia': float(np.sqrt(np.mean(delta ** 2))),
              'erro_absoluto_maximo': float(np.max(np.abs(delta))),
              'concordancia_classe_limiar_0_5': float(np.mean((values >= 0.5) == (reference >= 0.5))),
              'decisao_simulada': decision_disagreement(reference, values)}
    if weights is not None:
        result['decisao_ponderada_por_saldo_original'] = decision_disagreement(reference, values, weights)
    if POLICY_CUTOFF is not None:
        result['discordancia_aprovacao_no_limiar'] = float(np.mean((reference <= POLICY_CUTOFF) != (values <= POLICY_CUTOFF)))
    return result


def predictive(y, probabilities):
    if len(np.unique(y)) != 2:
        return {'metricas': 'não calculadas: a amostra precisa conter as duas classes'}
    return {'roc_auc': float(roc_auc_score(y, probabilities)),
            'average_precision': float(average_precision_score(y, probabilities)),
            'brier': float(brier_score_loss(y, probabilities)),
            'log_loss': float(log_loss(y, probabilities, labels=[0, 1]))}


def evaluate_batch(data, labels=None, include_fhe=False):
    reference, reference_time = timed_probability(baseline_model.predict_proba, data)
    clear, clear_time = timed_probability(lambda x: model.predict_proba(x, fhe='disable'), data)
    simulated, simulation_time = timed_probability(lambda x: model.predict_proba(x, fhe='simulate'), data)
    # A coluna 4 é o saldo original tratado pelo pré-processamento do próprio N01.
    weights = np.asarray(data[:, 4], dtype=float)
    if not np.isfinite(weights).all() or (weights < 0).any() or weights.sum() <= 0:
        weights = None
    summary = {'n': len(data), 'tempos_s': {'referencia': reference_time, 'ckks_disable': clear_time,
                                             'ckks_simulate': simulation_time},
               'ckks_disable_vs_referencia': fidelity(reference, clear, weights),
               'ckks_simulate_vs_referencia': fidelity(reference, simulated, weights)}
    if weights is None:
        summary['decisao_ponderada'] = 'não calculada: pesos inválidos ou ausentes'
    if POLICY_CUTOFF is not None:
        summary['limiar_de_aprovacao'] = POLICY_CUTOFF
    if labels is not None:
        summary['metricas_preditivas'] = {'referencia': predictive(labels, reference),
                                          'ckks_simulate': predictive(labels, simulated)}
    if include_fhe:
        encrypted, encrypted_time = timed_probability(lambda x: model.predict_proba(x, fhe='execute'), data)
        summary['tempos_s']['ckks_execute'] = encrypted_time
        summary['ckks_execute_vs_referencia'] = fidelity(reference, encrypted, weights)
        summary['ckks_execute_vs_simulate'] = fidelity(simulated, encrypted, weights)
        last_run = getattr(model, 'last_run_', {})
        if isinstance(last_run, dict):
            summary['tempos_fhe_por_etapa_s'] = {
                stage: float(last_run[stage]) for stage in ('encrypt', 'server', 'decrypt')
                if isinstance(last_run.get(stage), (int, float)) and np.isfinite(last_run[stage])
            }
    return summary

from IPython.display import Markdown, display


def format_value(value):
    if value is None:
        return '—'
    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        return f'{value:.9g}'
    return str(value).replace('|', r'\|').replace('\n', ' ')


def markdown_table(headers, rows):
    header = '| ' + ' | '.join(headers) + ' |'
    separator = '| ' + ' | '.join(['---'] * len(headers)) + ' |'
    body = ['| ' + ' | '.join(format_value(value) for value in row) + ' |' for row in rows]
    return '\n'.join([header, separator, *body])


def show_summary(summary, title):
    comparisons = {
        'ckks_disable_vs_referencia': 'CKKS em claro vs referência',
        'ckks_simulate_vs_referencia': 'CKKS simulado vs referência',
        'ckks_execute_vs_referencia': 'CKKS cifrado vs referência',
        'ckks_execute_vs_simulate': 'CKKS cifrado vs simulado',
    }
    items = [(label, summary[key]) for key, label in comparisons.items() if key in summary]
    lines = [f'### {title}', f"Observações avaliadas: **{summary['n']}**.",
             'Valores exibidos com até nove algarismos significativos; taxas são proporções entre 0 e 1.',
             '', '**Fidelidade numérica**', '']
    fidelity_rows = [(label, item.get('mae_vs_referencia'), item.get('rmse_vs_referencia'),
                      item.get('erro_absoluto_maximo'), item.get('concordancia_classe_limiar_0_5'))
                     for label, item in items]
    lines += [markdown_table(['Comparação', 'MAE', 'RMSE', 'Erro absoluto máximo', 'Concordância: p ≥ 0,5'], fidelity_rows),
              '', '**Discordância decisória ao variar o limiar**', '']
    decision_rows = [(label, item['decisao_simulada'].get('discordancia_maxima'),
                      item['decisao_simulada'].get('area_discordancia'),
                      item['decisao_simulada'].get('diferenca_area_vs_mae')) for label, item in items]
    lines += [markdown_table(['Comparação', 'Discordância máxima', 'Área', 'Área − MAE'], decision_rows), '']
    weighted_rows = [(label, item['decisao_ponderada_por_saldo_original'].get('discordancia_maxima'),
                      item['decisao_ponderada_por_saldo_original'].get('area_discordancia'),
                      item['decisao_ponderada_por_saldo_original'].get('diferenca_area_vs_mae'))
                     for label, item in items if 'decisao_ponderada_por_saldo_original' in item]
    if weighted_rows:
        lines += ['**Discordância ponderada pelo saldo original tratado**', '',
                  markdown_table(['Comparação', 'Discordância máxima', 'Área', 'Área − MAE ponderado'], weighted_rows), '']
    elif 'decisao_ponderada' in summary:
        lines += ['Discordância ponderada: ' + format_value(summary['decisao_ponderada']) + '.', '']
    if 'limiar_de_aprovacao' in summary:
        cutoff_rows = [(label, item.get('discordancia_aprovacao_no_limiar')) for label, item in items]
        lines += [f"**Discordância no limiar de aprovação {format_value(summary['limiar_de_aprovacao'])}**", '',
                  markdown_table(['Comparação', 'Discordância'], cutoff_rows), '']
    if 'metricas_preditivas' in summary:
        rows, notes = [], []
        labels = {'referencia': 'Referência', 'ckks_simulate': 'CKKS simulado'}
        for key, metrics in summary['metricas_preditivas'].items():
            label = labels.get(key, key)
            if 'metricas' in metrics:
                notes.append(f"{label}: {format_value(metrics['metricas'])}.")
            else:
                rows.append((label, metrics.get('roc_auc'), metrics.get('average_precision'),
                             metrics.get('brier'), metrics.get('log_loss')))
        lines += ['**Métricas preditivas**', '']
        if rows:
            lines += [markdown_table(['Modelo', 'ROC-AUC', 'Average precision', 'Brier', 'Log loss'], rows), '']
        lines.extend(notes)
        if notes:
            lines.append('')
    time_labels = {'referencia': 'Referência', 'ckks_disable': 'CKKS em claro',
                   'ckks_simulate': 'CKKS simulado', 'ckks_execute': 'Chamada FHE completa'}
    time_rows = [(time_labels.get(key, key), value) for key, value in summary['tempos_s'].items()]
    stage_labels = {'encrypt': 'Cifragem (pode incluir chaves)', 'server': 'Avaliação homomórfica local',
                    'decrypt': 'Decifragem'}
    time_rows.extend((stage_labels.get(key, key), value) for key, value in summary.get('tempos_fhe_por_etapa_s', {}).items())
    lines += ['**Tempos de chamadas e etapas (segundos)**', '', markdown_table(['Chamada ou etapa', 'Tempo (s)'], time_rows)]
    display(Markdown('\n'.join(lines)))


wide_summary = evaluate_batch(X_test[:wide_size], y_test[:wide_size])
show_summary(wide_summary, 'Avaliação ampliada')


In [ ]:
small_summary = evaluate_batch(X_test[:fhe_size], include_fhe=RUN_FHE_EXECUTE)
if not RUN_FHE_EXECUTE:
    print('FHE real não executado: RUN_FHE_EXECUTE está desativado.')
show_summary(small_summary, 'Lote de inferência FHE')


## 5. Interpretação e limitações

- No modo `run`, manifesto e hashes verificam a integridade dos arquivos, mas não substituem confiança na origem. No modo `reproducao_local`, os hashes identificam os arquivos atuais, porém a comparação com os hashes e o ambiente da execução original é necessária antes de afirmar que se trata dos mesmos dados e modelo.
- As métricas do lote ampliado precisam ter o mesmo tamanho e as mesmas observações para serem comparáveis às tabelas do artigo. O máximo de discordância sobre limiares não equivale à discordância de uma política específica.
- A simulação mostra o programa CKKS sem criptografia; apenas `execute` realiza inferência cifrada. Uma diferença entre referência e CKKS pode incluir limitação, quantização ou aproximação; a diferença entre simulação e execução não prova, por si só, a causa do erro.
- A primeira cifragem pode incluir geração de chaves. Os tempos por etapa são informados pela biblioteca, não incluem comunicação de rede e não somam necessariamente o tempo total. Repita medições antes de interpretá-las como benchmark e não compare diretamente máquinas ou arquiteturas diferentes.
- O N01 não exige explicitamente que a data do atraso ou do encerramento seja igual ou posterior ao primeiro pagamento; registros anteriores podem afetar elegibilidade ou rótulo. A reprodução mantém essa regra para comparabilidade, e seu impacto precisa ser avaliado.
- O alvo, o pré-processamento e a divisão temporal seguem o N01 exploratório; sua validade científica não é automaticamente comprovada por este notebook. A Freddie Mac contém operações já originadas, de modo que decisões de aprovação são apenas simulações retrospectivas.
- Não grave dados, modelos, predições, chaves ou saídas individuais no repositório. Limpe as saídas do notebook antes de qualquer commit e revise métricas agregadas antes de compartilhá-las.
